# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Camello, Zack Kenshin)_: 
- _Student No._: 2024-08825
- _Section_: THR-TX-1

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

### Report discussion

1. What were the basis linear equations used to set up the matrix in each question of Problem 2?
   - In problem 2, the system of linear equations were derived from the following two facts. First, we know that along the x-axis, the only forces along this direction are purely the Tension from both ropes along this axis. Because the box is motionless, we know that $T_{x1}$ and $T_{x2}$ altogether must equate to $0$. Thus from this, we get our first linear equation, $-T_{x1} + T_{x2} = 0$ ($T_{x1}$ is negative since it pulls to the left while $T_{x2}$ pulls to the right). Next, we know that along the y-axis, the tension from the rope pulls upwards, while the weight of the box pulls downwards. Altogether, we know that all three of these forces add up to $0$ as the box is not moving. With a little algebraic manipulation, we then arrive with our final equation $T_{y1} + T_{y2} = mg$ (Both $T_{y1}$ and $T_{y2}$ act upwards, while $mg$ is downwards which is why we can equate it like this). This then provides us with the matrix:
  
$$
\begin{bmatrix}
    \sin a & \sin b \\
    -\cos a & \cos b
\end{bmatrix}
\begin{bmatrix}
    T_1 \\
    T_2
\end{bmatrix}
=
\begin{bmatrix}
    mg \\
    0
\end{bmatrix}
$$

2. Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function
  - The function used to solve the matrix equations were ```np.linalg.solve``` for smaller matrices (parts of Problem 1 and 2), and ```solve_banded``` from ```scipy.linalg``` for the 10,000-junction portion of Problem 1. For ```np.linalg.solve```, the function calculates this by implementing LU decomposition, which is the matrix form of the commonly taught gaussian elimination method. In short, it solves the system of linear equations by separating the matrix into lower and upper half triangles. While ```solve_banded``` also uses LU decomposition, its input matrix is instead a compressed form of the original matrix that only includes non-zero diagonals. By converting the original $10000 \times 10000$ matrix into a much smaller $5 \times 10000$ matrix, the algorithm is able to perform LU decomposition with significantly more ease and speed. 

### Other comments

---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

In [2]:
def matrix(N):
    A = []
    w = []

    for i in range(N):
        row = [0] * N # Creates the basis of each row for the matrix which is just edited as the function goes on
        row[i] = 4 # ith term in the row is 4, as its all in the main diagonal (the first and last are 3 which get replaced later on)

        # These set the values for the i-1th, i-2th, i+1th, i+2th terms equal to -1
        if i - 1 >= 0: # stops this from looping around when i = 0 for example
            row[i - 1] = -1
        if i - 2 >= 0: 
            row[i - 2] = -1
        if i + 1 < N:  
            row[i + 1] = -1
        if i + 2 < N:  
            row[i + 2] = -1

        A.append(row) # Appends this row into the main matrix
        
        if i < 2:
            w.append(5) # Creates the 2 initial 5s in the solution matrix
        else:
            w.append(0) # Adds the remaining 0s to the solution matrix
                
    A[0][0] = 3 # Replaces the first term in main diagonal with 3
    A[-1][-1] = 3 # Replaces the second term in main diagonal with 3
    
    return A, w

A, w = matrix(6)
print(A)
print(w)

Voltages = np.linalg.solve(A, w)
print(Voltages)

[[3, -1, -1, 0, 0, 0], [-1, 4, -1, -1, 0, 0], [-1, -1, 4, -1, -1, 0], [0, -1, -1, 4, -1, -1], [0, 0, -1, -1, 4, -1], [0, 0, 0, -1, -1, 3]]
[5, 5, 0, 0, 0, 0]
[3.7254902  3.43137255 2.74509804 2.25490196 1.56862745 1.2745098 ]


In [23]:
from scipy.linalg import solve_banded

N = 10000

A = np.zeros((5, N)) # Creates a matrix of 5 rows and N columns of JUST 0s
A[0, 2:] = -1 # First top diagonal of the original matrix, we start at 2 since the diagonal starts at the 3rd number
A[1, 1:] = -1 # Second top diagonal of the original matrix, starts at 1 since the diagonal starts at 2nd number
A[2] = 4 # The main diagonal, which fills the entire diagonal with just 4
A[3, :-1] = -1 # First lower diagonal, fills entire diagonal with -1 except last entry
A[4, :-2] = -1 # Second lower diagonal, fills entire diagonal with -1 except the last two entries

A[2, 0] = 3 # Replaces first of the main diagonal of the matrix with 3
A[2, -1] = 3 # Replaces last of the main diagonal of the matrix with 3

w = np.zeros((N, 1)) # Creates a column of 0s with N rows
w[0] = 5 # Replaces first value of the w matrix with 5
w[1] = 5 # Replaces second value fo the w matrix with 5

Voltages = solve_banded((2, 2), A, w) # Using 2 upper and lower diagonals, and the banded matrix, with the solution matrix w

print(f"First 5 voltages: {Voltages[:5]}")
print(f"Last 5 voltages: {Voltages[-5:]}")


First 5 voltages: [[4.99888228]
 [4.99861842]
 [4.99802841]
 [4.99756299]
 [4.99704997]]
Last 5 voltages: [[0.00295003]
 [0.00243701]
 [0.00197159]
 [0.00138158]
 [0.00111772]]


#### Problem 1 code summary

The algorithm above calculates the Voltage values at each junction given $N$ amount of junctions. It does so by initially creating a row of $N$ amount of zeroes, replaces the ``ith`` value with $4$, and the ```i - 1, i -2, i + 1, and i + 2``` slots with $-1$. While that loop occurs, a solution matrix (w) is created with a value of $5$ when $i<2$, and $0$ for the rest of the entries. The top left and bottom right corners of the matrix are then replaced with a $3$, and the matrices are input and solved through ```np.linalg.solve```. A similar approach is done for the $10000$-junction portion, in which the non-zero diagonals are placed row by row to create a compressed matrix. This matrix is then put through ```solve_banded``` of the ```scipy.linalg``` library to solve for the voltages.

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.

<del> 
Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right
</del>

In [26]:
from custom import tension

angle_a = float(input("Desired alpha angle (degrees): "))
angle_b = float(input("Desired beta angle (degrees): "))
mass = float(input("Desired mass (kilograms): "))

tension = tension(angle_a, angle_b, mass)
print("T1: ", tension[0], "N",  " and T2: ", tension[1], "N")

Desired alpha angle (degrees):  45
Desired beta angle (degrees):  45
Desired mass (kilograms):  10


T1:  69.36717523440032 N  and T2:  69.36717523440032 N


#### Problem 2 code summary

The algorithm above simply uses the matrices, 

$
\begin{bmatrix}
    \sin a & \sin b \\
    -\cos a & \cos b
\end{bmatrix}
\begin{bmatrix}
    T_1 \\
    T_2
\end{bmatrix}
=
\begin{bmatrix}
    mg \\
    0
\end{bmatrix}
$

given the specified angles and mass, and calculates $T_{1}$ and $T_{2}$ using np.linalg.solve. 

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [4]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])


In [5]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [6]:
ans = np.linalg.solve(sigma_x, sigma_y)
ans = np.linalg.inv(A) @ B


In [7]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

(array([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [-0.        ,  1.        ,  0.        ],
        [-0.5       , -0.16666667,  1.        ]]),
 array([[-2.        , -1.        ,  3.        ],
        [ 0.        ,  3.        ,  5.        ],
        [ 0.        ,  0.        ,  4.33333333]]))

### Importing functions from a python file

In [15]:
import custom as sP

In [16]:
testMat = sP.createZerosArray(4,5)

In [17]:
testMat[1] = [1,2,3,4,5]

In [18]:
testMat

array([[0., 0., 0., 0., 0.],
       [1., 2., 3., 4., 5.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

### np.roll

In [ ]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
#x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)